In [3]:
"""
Applied Python-Based Data Preprocessing and Analysis Pipeline
Libraries: Pandas, NumPy
Compatibility: Python 3.9+ | Pandas 2.x, 3.x, 4.x | NumPy 1.x, 2.x
"""

import numpy as np
import pandas as pd


# ==========================================
# 1. SYNTHETIC DIRTY DATASET GENERATOR
# ==========================================
def generate_sample_dirty_dataset(n_rows: int = 150) -> pd.DataFrame:
    """
    Generates a realistic, dirty dataset with missing values, string noise,
    outliers, invalid types, and datetime variations.
    """
    np.random.seed(42)

    # 1. Customer IDs
    customer_ids = np.arange(1001, 1001 + n_rows)

    # 2. Age (with NaNs, negative values, and extreme outliers)
    age = np.random.normal(loc=35, scale=12, size=n_rows).round()
    age[np.random.choice(n_rows, size=10, replace=False)] = np.nan
    age[np.random.choice(n_rows, size=3, replace=False)] = -5   # Unrealistic / dirty data
    age[np.random.choice(n_rows, size=2, replace=False)] = 150  # Extreme outlier

    # 3. Income (positive skew, NaNs, and extreme high values)
    income = np.random.exponential(scale=50000, size=n_rows) + 20000
    income[np.random.choice(n_rows, size=8, replace=False)] = np.nan
    income[0] = 1_500_000  # High outlier

    # 4. Membership Tier (Categorical with inconsistent casing and whitespaces)
    tiers = [" Bronze ", "SILVER", "gold", "Gold", "Platinum", " bronze", None]
    membership = np.random.choice(tiers, size=n_rows, p=[0.3, 0.25, 0.15, 0.15, 0.1, 0.03, 0.02])

    # 5. Purchase Date (as strings with mixed formats & dirty string entry)
    base_date = pd.Timestamp("2025-01-01")
    date_offsets = [pd.Timedelta(days=int(d)) for d in np.random.randint(0, 365, size=n_rows)]
    dates = [(base_date + offset).strftime("%Y-%m-%d") for offset in date_offsets]
    dates[5] = "invalid_date_entry"

    # 6. Purchase Amount (Continuous numeric)
    purchase_amt = np.random.gamma(shape=2.0, scale=100.0, size=n_rows)

    # 7. Customer Satisfaction Score (Ordinal 1 to 5 with missing values)
    satisfaction = np.random.choice([1, 2, 3, 4, 5, np.nan], size=n_rows, p=[0.1, 0.15, 0.3, 0.3, 0.1, 0.05])

    df = pd.DataFrame({
        "customer_id": customer_ids,
        "age": age,
        "income": income,
        "membership_tier": membership,
        "signup_date": dates,
        "purchase_amount": purchase_amt,
        "satisfaction_score": satisfaction
    })

    return df


# ==========================================
# 2. PREPROCESSING & CLEANING PIPELINE
# ==========================================
class DataPreprocessor:
    """Modular Data Preprocessing Pipeline utilizing Pandas and NumPy."""

    def __init__(self, df: pd.DataFrame):
        self.raw_df = df.copy()
        self.df = df.copy()

    def inspect_data(self) -> dict:
        """Performs initial profiling and health checks on the dataset."""
        return {
            "shape": self.df.shape,
            "missing_values": self.df.isnull().sum().to_dict(),
            "missing_pct": (self.df.isnull().mean() * 100).round(2).to_dict(),
            "dtypes": self.df.dtypes.astype(str).to_dict(),
            "duplicates": int(self.df.duplicated().sum())
        }

    def clean_text_and_categories(self, col: str) -> "DataPreprocessor":
        """Strips whitespace, unifies title casing, and normalizes missing representations."""
        if col in self.df.columns:
            self.df[col] = (
                self.df[col]
                .astype(str)
                .str.strip()
                .str.title()
                .replace({"None": np.nan, "Nan": np.nan, "<Na>": np.nan, "": np.nan})
            )
        return self

    def parse_datetime(self, col: str) -> "DataPreprocessor":
        """Converts strings to datetime (coercing invalid values to NaT) and extracts temporal features."""
        if col in self.df.columns:
            self.df[col] = pd.to_datetime(self.df[col], errors="coerce")
            self.df[f"{col}_year"] = self.df[col].dt.year
            self.df[f"{col}_month"] = self.df[col].dt.month
            self.df[f"{col}_dayofweek"] = self.df[col].dt.day_name()
        return self

    def handle_missing_values(self) -> "DataPreprocessor":
        """
        Imputes missing values:
        - Numeric columns: Median imputation
        - Categorical & String columns: Mode imputation (compatible with Pandas 2.x, 3.x, and 4.x)
        """
        # 1. Numeric imputation
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if self.df[col].isnull().any():
                median_val = self.df[col].median()
                self.df[col] = self.df[col].fillna(median_val)

        # 2. Categorical & String imputation (explicitly includes 'str' and 'string' to prevent Pandas4Warning)
        cat_cols = self.df.select_dtypes(include=["object", "category", "str", "string"]).columns
        for col in cat_cols:
            if self.df[col].isnull().any():
                mode_val = self.df[col].mode()[0] if not self.df[col].mode().empty else "Unknown"
                self.df[col] = self.df[col].fillna(mode_val)

        return self

    def handle_outliers_iqr(self, col: str, factor: float = 1.5, cap: bool = True) -> "DataPreprocessor":
        """
        Detects and caps outliers using the Interquartile Range (IQR) method.
        If cap=True, bounds are capped using np.clip without dropping rows.
        """
        if col in self.df.columns and pd.api.types.is_numeric_dtype(self.df[col]):
            q25 = self.df[col].quantile(0.25)
            q75 = self.df[col].quantile(0.75)
            iqr = q75 - q25
            lower_bound = q25 - (factor * iqr)
            upper_bound = q75 + (factor * iqr)

            if cap:
                self.df[col] = np.clip(self.df[col], lower_bound, upper_bound)
            else:
                self.df = self.df[(self.df[col] >= lower_bound) & (self.df[col] <= upper_bound)]
        return self

    def feature_engineering(self) -> "DataPreprocessor":
        """Generates derived features via fast vectorized NumPy functions."""
        # 1. Multi-conditional segmentation using np.select
        conditions = [
            (self.df["income"] >= 80000) & (self.df["purchase_amount"] >= 250),
            (self.df["income"] >= 50000) | (self.df["purchase_amount"] >= 150),
            (self.df["income"] < 50000) & (self.df["purchase_amount"] < 150)
        ]
        choices = ["High Value", "Medium Value", "Budget"]
        self.df["customer_segment"] = np.select(conditions, choices, default="Standard")

        # 2. Log-transformation for right-skewed numerical columns
        self.df["log_income"] = np.log1p(np.maximum(0, self.df["income"]))
        self.df["log_purchase_amt"] = np.log1p(np.maximum(0, self.df["purchase_amount"]))

        # 3. Continuous feature binning using pd.cut
        self.df["age_group"] = pd.cut(
            self.df["age"],
            bins=[0, 25, 40, 60, 100],
            labels=["Young Adult", "Adult", "Middle-Aged", "Senior"],
            right=False
        )
        return self

    def scale_features(self, cols: list[str], method: str = "standard") -> "DataPreprocessor":
        """
        Scales numeric features using pure NumPy operations:
        - 'standard': Z-score normalization ((x - mean) / std)
        - 'minmax': Range scaling ((x - min) / (max - min))
        """
        for col in cols:
            if col in self.df.columns:
                arr = self.df[col].to_numpy()
                if method == "standard":
                    mean, std = np.mean(arr), np.std(arr)
                    self.df[f"{col}_std_scaled"] = (arr - mean) / (std if std != 0 else 1)
                elif method == "minmax":
                    min_val, max_val = np.min(arr), np.max(arr)
                    denom = max_val - min_val
                    self.df[f"{col}_minmax_scaled"] = (arr - min_val) / (denom if denom != 0 else 1)
        return self

    def get_processed_data(self) -> pd.DataFrame:
        """Returns the final cleaned DataFrame."""
        return self.df


# ==========================================
# 3. EXPLORATORY DATA ANALYSIS (EDA)
# ==========================================
class DataAnalyzer:
    """Performs aggregations, correlation analysis, and statistical reporting."""

    def __init__(self, df: pd.DataFrame):
        self.df = df

    def compute_summary_stats(self) -> pd.DataFrame:
        """Returns descriptive statistics for all numeric features."""
        return self.df.describe().T

    def segment_analysis(self) -> pd.DataFrame:
        """Computes grouped summary metrics across segments and membership tiers."""
        grouped = self.df.groupby(["customer_segment", "membership_tier"]).agg(
            total_customers=("customer_id", "count"),
            avg_income=("income", "mean"),
            median_income=("income", "median"),
            avg_purchase=("purchase_amount", "mean"),
            total_revenue=("purchase_amount", "sum"),
            avg_satisfaction=("satisfaction_score", "mean")
        ).round(2)
        return grouped

    def compute_correlation_matrix(self) -> pd.DataFrame:
        """Calculates Pearson correlation matrix for numeric columns."""
        numeric_df = self.df.select_dtypes(include=[np.number])
        return numeric_df.corr().round(3)

    def print_insights(self):
        """Displays high-level analytical findings."""
        print("\n" + "=" * 70)
        print("                 EXPLORATORY DATA ANALYSIS SUMMARY")
        print("=" * 70)

        print("\n--- Key Metrics by Customer Segment & Membership Tier ---")
        print(self.segment_analysis())

        print("\n--- Feature Correlation with Purchase Amount ---")
        corr = self.compute_correlation_matrix()
        if "purchase_amount" in corr.columns:
            print(corr["purchase_amount"].sort_values(ascending=False))


# ==========================================
# 4. MAIN EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    print("[1] Generating raw dirty dataset...")
    raw_data = generate_sample_dirty_dataset(n_rows=150)
    print(f"Raw Data Shape: {raw_data.shape}\n")

    print("[2] Initial Data Profiling...")
    pipeline = DataPreprocessor(raw_data)
    initial_health = pipeline.inspect_data()
    print("Detected Missing Values:")
    for col, count in initial_health["missing_values"].items():
        if count > 0:
            print(f"  • {col:<20}: {count:>3} ({initial_health['missing_pct'][col]}%)")

    print("\n[3] Executing Preprocessing & Feature Pipeline...")
    clean_df = (
        pipeline
        .clean_text_and_categories("membership_tier")
        .parse_datetime("signup_date")
        .handle_missing_values()
        .handle_outliers_iqr("age", factor=1.5, cap=True)
        .handle_outliers_iqr("income", factor=1.5, cap=True)
        .feature_engineering()
        .scale_features(cols=["income", "purchase_amount"], method="standard")
        .scale_features(cols=["income", "purchase_amount"], method="minmax")
        .get_processed_data()
    )

    print("\n[4] Cleaned & Preprocessed Sample:")
    display_cols = [
        "customer_id", "age", "age_group", "membership_tier", 
        "income", "purchase_amount", "customer_segment"
    ]
    print(clean_df[display_cols].head(8))

    print("\n[5] Running Exploratory Data Analysis...")
    analyzer = DataAnalyzer(clean_df)
    analyzer.print_insights()

[1] Generating raw dirty dataset...
Raw Data Shape: (150, 7)

[2] Initial Data Profiling...
Detected Missing Values:
  • age                 :  10 (6.67%)
  • income              :   8 (5.33%)
  • membership_tier     :   2 (1.33%)
  • satisfaction_score  :   7 (4.67%)

[3] Executing Preprocessing & Feature Pipeline...

[4] Cleaned & Preprocessed Sample:
   customer_id   age    age_group membership_tier         income  \
0         1001  41.0  Middle-Aged            Gold  155597.843528   
1         1002  33.0        Adult            Gold   58215.650027   
2         1003  43.0  Middle-Aged          Bronze   89296.871135   
3         1004  53.0  Middle-Aged          Silver  142187.708795   
4         1005  32.0        Adult          Bronze   63991.857127   
5         1006  32.0        Adult          Silver   84753.415332   
6         1007  54.0  Middle-Aged          Bronze   90751.402814   
7         1008  44.0  Middle-Aged            Gold   43728.747031   

   purchase_amount customer_seg